# BirdCLEF 2026 - Submit Candidate 2: Exp60 High-OOF Taxon Prior

Kaggle-ready submission notebook adapted from `exp46_submit_kaggle.ipynb` using the best high-OOF settings from `exp60_taxon_dual_power`.

**Why this one:** it is the strongest non-oracle OOF branch so far (`OOF padded cmAP=0.95742`) and differs materially from exp22 through larger MLP probes, taxon-aware priors, dual event/texture file-confidence power, and a 90/10 0s/2.5s TTA blend.

It trains on the full available training soundscape labels, applies test-time inference, and writes `/kaggle/working/submission.csv`.


In [ ]:
# Kaggle input discovery and imports
import concurrent.futures
import gc
import json
import os
import re
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
LOCAL_ROOT = Path.cwd().resolve()
if not (LOCAL_ROOT / 'data').exists() and (LOCAL_ROOT.parent / 'data').exists():
    LOCAL_ROOT = LOCAL_ROOT.parent
IS_KAGGLE = KAGGLE_INPUT.exists()
WORK_DIR = KAGGLE_WORKING if IS_KAGGLE else LOCAL_ROOT
CACHE_DIR = WORK_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
N_WINDOWS = 12
TTA_OFFSET_SEC = 2.5
BATCH_FILES = 8

BEST_CFG = dict(lambda_event=2.2, lambda_texture=9.0, power_event=0.90, power_texture=1.20, alpha_event=0.10, alpha_texture=0.30, wp=0.02, wm=0.56, wpr=0.42, tta_w0=0.90)

CFG = dict(
    d_model=128, d_state=16, n_ssm_layers=2, dropout=0.15,
    n_sites=20, meta_dim=16,
    n_epochs=30, lr=5e-4, weight_decay=1e-3, patience=8,
    focal_gamma=1.5, distill_weight=0.00, pos_weight_cap=25.0,
    pca_dim=128, mlp_hidden=(256, 128), min_pos=3, alpha_blend=0.45,
)

CACHE_META_0 = CACHE_DIR / 'exp60_perch_meta.csv'
CACHE_NPZ_0 = CACHE_DIR / 'exp60_perch_arrays.npz'
SUBMISSION_PATH = WORK_DIR / 'submission.csv'
SUMMARY_PATH = CACHE_DIR / 'exp60_kaggle_submit_summary.json'

_WALL_START = time.time()
print('IS_KAGGLE:', IS_KAGGLE)
print('WORK_DIR:', WORK_DIR)
if IS_KAGGLE:
    print('/kaggle/input top-level:', sorted(p.name for p in KAGGLE_INPUT.iterdir())[:30])


In [ ]:
def first_existing(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None


def search_one(root, patterns):
    root = Path(root)
    if not root.exists():
        return None
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


DATA_DIR = first_existing([
    '/kaggle/input/birdclef-2026',
    '/kaggle/input/competitions/birdclef-2026',
    LOCAL_ROOT / 'data',
])
if DATA_DIR is None and IS_KAGGLE:
    labels_file = search_one(KAGGLE_INPUT, ['train_soundscapes_labels.csv'])
    DATA_DIR = labels_file.parent if labels_file else None
if DATA_DIR is None:
    raise FileNotFoundError('Could not find BirdCLEF data directory with train_soundscapes_labels.csv')

PERCH_ONNX_PATH = first_existing([
    LOCAL_ROOT / 'data/models/perch/perch_v2_no_dft.onnx',
])
if PERCH_ONNX_PATH is None and IS_KAGGLE:
    PERCH_ONNX_PATH = search_one(KAGGLE_INPUT, ['perch_v2_no_dft.onnx', '*.onnx'])

PERCH_LABELS_PATH = first_existing([
    LOCAL_ROOT / 'data/models/perch_tf/assets/labels.csv',
])
if PERCH_LABELS_PATH is None and IS_KAGGLE:
    label_hits = sorted(KAGGLE_INPUT.rglob('labels.csv'))
    asset_hits = [p for p in label_hits if 'asset' in str(p).lower() or 'perch' in str(p).lower()]
    PERCH_LABELS_PATH = (asset_hits or label_hits)[0] if label_hits else None

if PERCH_ONNX_PATH is None:
    raise FileNotFoundError('Attach a Kaggle dataset containing perch_v2_no_dft.onnx')
if PERCH_LABELS_PATH is None:
    raise FileNotFoundError('Attach a Kaggle dataset/model containing Perch assets/labels.csv')

try:
    import onnxruntime as ort
    print('ONNX Runtime already available')
except Exception as import_error:
    wheel = search_one(KAGGLE_INPUT if IS_KAGGLE else LOCAL_ROOT, ['onnxruntime-*.whl'])
    if wheel is None:
        raise ModuleNotFoundError(
            'onnxruntime is required. Attach a Kaggle input dataset containing an onnxruntime-*.whl file; '
            'internet installs are intentionally disabled.'
        ) from import_error
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', str(wheel)], check=True)
    import onnxruntime as ort
    print('ONNX Runtime installed from attached wheel:', wheel)

TRAIN_SOUNDSCAPE_DIR = DATA_DIR / 'train_soundscapes'
TEST_SOUNDSCAPE_DIR = DATA_DIR / 'test_soundscapes'

print('DATA_DIR:', DATA_DIR)
print('PERCH_ONNX_PATH:', PERCH_ONNX_PATH)
print('PERCH_LABELS_PATH:', PERCH_LABELS_PATH)
print('onnxruntime:', ort.__version__)
print('TRAIN_SOUNDSCAPE_DIR exists:', TRAIN_SOUNDSCAPE_DIR.exists())
print('TEST_SOUNDSCAPE_DIR exists:', TEST_SOUNDSCAPE_DIR.exists())

# Data loading
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].astype(str).tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()
assert taxonomy['primary_label'].astype(str).tolist() == PRIMARY_LABELS, 'taxonomy order differs from sample_submission'

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')


def parse_fname(name):
    m = FNAME_RE.match(Path(str(name)).name)
    if not m:
        return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}


def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)


sc = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id'] = sc['filename'].str.replace('.ogg', '', regex=False) + '_' + sc['end_sec'].astype(str)
_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby('filename').size()
LABEL_WINDOWS = int(windows_per_file.mode().iloc[0])
full_files = sorted(windows_per_file[windows_per_file == LABEL_WINDOWS].index.tolist())
sc['fully_labeled'] = sc['filename'].isin(full_files)
full_rows = sc[sc['fully_labeled']].sort_values(['filename', 'end_sec']).reset_index(drop=False)

print(f'Classes: {N_CLASSES} | Fully-labeled files for sequence training: {len(full_files)}')
print(f'Total labeled rows for priors: {len(sc)}')
print('Sample submission shape:', sample_sub.shape)

In [ ]:
# Load Perch ONNX + label mapping
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
ONNX_SESSION = ort.InferenceSession(str(PERCH_ONNX_PATH), sess_options=_so, providers=['CPUExecutionProvider'])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print('Perch input:', ONNX_INPUT_NAME, ONNX_SESSION.get_inputs()[0].shape)
print('Perch outputs:', list(ONNX_OUT_MAP))

bc_labels_raw = pd.read_csv(PERCH_LABELS_PATH)
if 'inat2024_fsd50k' in bc_labels_raw.columns:
    sci_col = 'inat2024_fsd50k'
elif 'scientific_name' in bc_labels_raw.columns:
    sci_col = 'scientific_name'
else:
    sci_col = bc_labels_raw.columns[0]
bc_labels = bc_labels_raw[[sci_col]].copy()
bc_labels.columns = ['scientific_name']
bc_labels['bc_index'] = np.arange(len(bc_labels), dtype=np.int32)

NO_LABEL = len(bc_labels)
mapping = taxonomy.merge(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index('primary_label')['bc_index']
BC_INDICES = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)

proxy_map = {}
for _, row in taxonomy[taxonomy['primary_label'].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits = bc_labels[bc_labels['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)]
    if len(hits) > 0:
        proxy_map[label_to_idx[row['primary_label']]] = hits['bc_index'].astype(int).tolist()
proxy_map = {k: v for k, v in proxy_map.items() if CLASS_NAME_MAP.get(PRIMARY_LABELS[k]) in {'Amphibia', 'Insecta', 'Aves'}}


def run_perch_waves(waves):
    waves = np.asarray(waves, dtype=np.float32)
    outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: waves})
    emb_idx = ONNX_OUT_MAP.get('embedding', 0)
    label_idx = ONNX_OUT_MAP.get('label', 1 if len(outs) > 1 else 0)
    return outs[emb_idx].astype(np.float32), outs[label_idx].astype(np.float32)


def comp_scores_from_logits(logits):
    scores = np.zeros((len(logits), N_CLASSES), dtype=np.float32)
    scores[:, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
    for pos_idx, bc_idxs in proxy_map.items():
        scores[:, pos_idx] = logits[:, np.array(bc_idxs, dtype=np.int32)].max(axis=1)
    return scores

print(f'Perch loaded. Mapped: {MAPPED_MASK.sum()}/{N_CLASSES} | Proxy targets: {len(proxy_map)}')

In [ ]:
# Perch feature generation for standard and 2.5s shifted windows
SCORE_KEYS = ['scores', 'sc', 'logits', 'arr_0']
EMB_KEYS = ['embs', 'emb', 'embeddings', 'arr_1']


def _pick_array(arr, candidates, ncols):
    for k in candidates:
        if k in arr.files:
            return arr[k]
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == ncols:
            return v
    raise KeyError(f'Keys {candidates} not found. Available: {arr.files}')


def read_windowed_audio(path, offset_sec=0.0):
    y, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    offset_samples = int(round(offset_sec * SR))
    needed = offset_samples + N_WINDOWS * WINDOW_SAMPLES
    if len(y) < needed:
        y = np.pad(y, (0, needed - len(y)))
    y = y[offset_samples:offset_samples + N_WINDOWS * WINDOW_SAMPLES]
    return y.reshape(N_WINDOWS, WINDOW_SAMPLES).astype(np.float32)


def run_perch_windows(paths, offset_sec=0.0, batch_files=BATCH_FILES, verbose=True, row_id_suffix=''):
    paths = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.zeros(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs = np.zeros((n_rows, 1536), dtype=np.float32)

    wr = 0
    ranges = range(0, len(paths), batch_files)
    itr = tqdm(ranges, desc=f'Perch offset {offset_sec:g}s') if verbose else ranges
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        next_paths = paths[:batch_files]
        future_audio = [executor.submit(read_windowed_audio, p, offset_sec) for p in next_paths]
        for start in itr:
            batch_paths = next_paths
            batch_audio = [f.result() for f in future_audio]
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths = paths[next_start:next_start + batch_files]
                future_audio = [executor.submit(read_windowed_audio, p, offset_sec) for p in next_paths]

            x = np.stack(batch_audio).reshape(len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES)
            br = wr
            for path in batch_paths:
                meta = parse_fname(path.name)
                stem = path.stem
                row_ids[wr:wr + N_WINDOWS] = [f'{stem}{row_id_suffix}_{t}' for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites[wr:wr + N_WINDOWS] = meta['site']
                hours[wr:wr + N_WINDOWS] = meta['hour_utc']
                wr += N_WINDOWS

            emb, logits = run_perch_waves(x)
            scores[br:wr] = comp_scores_from_logits(logits)
            embs[br:wr] = emb
            del x, emb, logits, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours})
    return meta_df, scores, embs


def load_or_build_train_features():
    if CACHE_META_0.exists() and CACHE_NPZ_0.exists():
        print('Loading standard Perch cache')
        meta_0 = pd.read_csv(CACHE_META_0)
        arr_0 = np.load(CACHE_NPZ_0)
        sc_0 = _pick_array(arr_0, SCORE_KEYS, N_CLASSES).astype(np.float32)
        emb_0 = _pick_array(arr_0, EMB_KEYS, 1536).astype(np.float32)
    else:
        print(f'Building standard Perch cache from {len(full_files)} full-label files')
        train_paths = [TRAIN_SOUNDSCAPE_DIR / fn for fn in full_files]
        meta_0, sc_0, emb_0 = run_perch_windows(train_paths, offset_sec=0.0, row_id_suffix='')
        meta_0.to_csv(CACHE_META_0, index=False)
        np.savez(CACHE_NPZ_0, scores=sc_0.astype(np.float32), embs=emb_0.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))

    return meta_0, sc_0, emb_0


meta_tr, sc_0, emb_0 = load_or_build_train_features()
row_id_to_index = full_rows.set_index('row_id')['index']
Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr['row_id']].to_numpy()]
print(f'sc_0={sc_0.shape} emb_0={emb_0.shape}')
print('Y_FULL_aligned:', Y_FULL_aligned.shape)

In [ ]:
# Metrics + loss
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def focal_bce(logits, targets, gamma=1.5, pos_weight=None):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    p_t = torch.exp(-bce)
    return ((1 - p_t) ** gamma * bce).mean()

print('Metrics ready')

In [ ]:
# Post-processing with configurable hyperparameters

def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)
    site_keys = sorted(sc_df['site'].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = sc_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)
    hour_keys = sorted(sc_df['hour_utc'].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = sc_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)
    return {'global_p': global_p,
            'site_to_i': site_to_i, 'site_p': site_p, 'site_n': site_n,
            'hour_to_i': hour_to_i, 'hour_p': hour_p, 'hour_n': hour_n}

def apply_prior(scores, sites, hours, tables, lambda_prior=0.10):
    eps = 1e-4; out = scores.copy()
    p   = np.tile(tables['global_p'], (len(scores), 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1 - w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables['site_p'][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.10):
    result = probs.copy(); view = probs.reshape(-1, n_windows, probs.shape[1])
    out    = result.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0 if 0 < t < n_windows-1 else \
              (view[:, t, :] + view[:, t+1, :]) / 2.0 if t == 0 else \
              (view[:, t-1, :] + view[:, t, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.05):
    view = probs.reshape(-1, n_windows, probs.shape[1])
    top_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_mean + 1e-8, power)).reshape(probs.shape)

TEXTURE_TAXA = {'Amphibia', 'Insecta'}
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, lbl in enumerate(PRIMARY_LABELS):
    temperatures[ci] = 0.95 if CLASS_NAME_MAP.get(lbl, 'Aves') in TEXTURE_TAXA else 1.10

def postprocess(logits, power=0.05, base_alpha=0.10):
    p = sigmoid(logits / temperatures[None, :])
    p = file_confidence_scale(p, power=power)
    p = adaptive_delta_smooth(p, base_alpha=base_alpha)
    return np.clip(p, 0.0, 1.0)

# Per-class texture mask (True for Amphibia and Insecta)
TEXTURE_MASK = np.array([CLASS_NAME_MAP.get(lbl, 'Aves') in TEXTURE_TAXA for lbl in PRIMARY_LABELS])

def apply_prior_taxon(scores, sites, hours, tables, lambda_event=0.30, lambda_texture=0.60,
                       n_smooth_event=8.0, n_smooth_texture=8.0):
    """Apply site/hour prior with per-taxon lambda and smoothing."""
    eps = 1e-4; out = scores.copy()
    p = np.tile(tables['global_p'], (len(scores), 1))
    # Build per-class n_smooth array
    n_smooth_arr = np.where(TEXTURE_MASK, n_smooth_texture, n_smooth_event).astype(np.float32)
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1 - w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w_arr = ns / (ns + n_smooth_arr)  # per-class smoothing
            p[i] = w_arr * tables['site_p'][j] + (1 - w_arr) * p[i]
    p = np.clip(p, eps, 1 - eps)
    prior_logit = (np.log(p) - np.log1p(-p)).astype(np.float32)
    lambdas = np.where(TEXTURE_MASK, lambda_texture, lambda_event).astype(np.float32)
    out += lambdas[None, :] * prior_logit
    return out.astype(np.float32)

def adaptive_delta_smooth_taxon(probs, n_windows=N_WINDOWS, alpha_event=0.10, alpha_texture=0.10):
    """Temporal smoothing with per-taxon alpha (texture species get heavier smoothing)."""
    result = probs.copy(); view = probs.reshape(-1, n_windows, probs.shape[1])
    out = result.reshape(-1, n_windows, probs.shape[1])
    base_alpha_arr = np.where(TEXTURE_MASK, alpha_texture, alpha_event).astype(np.float32)
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha_arr[None, :] * (1.0 - conf)
        nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0 if 0 < t < n_windows-1 else \
              (view[:, t, :] + view[:, t+1, :]) / 2.0 if t == 0 else \
              (view[:, t-1, :] + view[:, t, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

def postprocess_taxon(logits, power=0.40, alpha_event=0.10, alpha_texture=0.10):
    p = sigmoid(logits / temperatures[None, :])
    p = file_confidence_scale(p, power=power)
    p = adaptive_delta_smooth_taxon(p, alpha_event=alpha_event, alpha_texture=alpha_texture)
    return np.clip(p, 0.0, 1.0)

def file_confidence_scale_dual(probs, n_windows=N_WINDOWS, top_k=2, power_event=1.0, power_texture=1.0):
    view = probs.reshape(-1, n_windows, probs.shape[1])
    top_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    powers = np.where(TEXTURE_MASK, power_texture, power_event).astype(np.float32)
    return (view * np.power(top_mean + 1e-8, powers[None, None, :])).reshape(probs.shape)

def postprocess_taxon_dual_power(logits, power_event=0.90, power_texture=1.20,
                                 alpha_event=0.10, alpha_texture=0.30):
    p = sigmoid(logits / temperatures[None, :])
    p = file_confidence_scale_dual(p, power_event=power_event, power_texture=power_texture)
    p = adaptive_delta_smooth_taxon(p, alpha_event=alpha_event, alpha_texture=alpha_texture)
    return np.clip(p, 0.0, 1.0)

prior_tables = build_prior_tables(sc, Y_SC)

print('Helpers ready')


In [ ]:
# MLP probes (identical to exp16)

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / Y.shape[0]
    weights   = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_seq_features(scores_col, n_windows=N_WINDOWS, threshold=0.0):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]],  axis=1).reshape(-1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    min_  = np.repeat(x.min(axis=1),  n_windows)
    # Rank of current score within file (0=lowest, 1=highest)
    ranks = np.argsort(np.argsort(x, axis=1), axis=1).astype(np.float32) / max(n_windows - 1, 1)
    rank  = ranks.reshape(-1)
    # Percentiles
    p25   = np.repeat(np.percentile(x, 25, axis=1), n_windows)
    p75   = np.repeat(np.percentile(x, 75, axis=1), n_windows)
    # Window position (0-1 within file)
    win_pos = np.tile(np.linspace(0, 1, n_windows), x.shape[0])
    # Difference from previous window (0 at start)
    x_prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    diff    = (x - x_prev).reshape(-1)
    # Activity features
    count_act = np.repeat((x > threshold).mean(axis=1), n_windows)
    median_   = np.repeat(np.median(x, axis=1), n_windows)
    above_med = (scores_col > median_).astype(np.float32)
    score_rng = np.repeat(x.max(axis=1) - x.min(axis=1), n_windows)
    return prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_act, above_med, score_rng

def train_mlp_probes(emb, scores_raw, Y, pca_dim=64, alpha_blend=0.25, min_pos=5):
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f'  PCA: {emb.shape} → {Z.shape}  ({pca.explained_variance_ratio_.sum():.2%})')
    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models  = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    MAX_ROWS = 2000
    for ci in active:
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_act, above_med, score_rng = build_seq_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       prev[:, None], next_[:, None],
                       mean[:, None], max_[:, None], std[:, None],
                       min_[:, None], rank[:, None],
                       p25[:, None], p75[:, None], win_pos[:, None], diff[:, None],
                       count_act[:, None], above_med[:, None], score_rng[:, None]])
        n_pos  = int(y.sum()); pos_idx = np.where(y == 1)[0]
        w      = float(class_weights[ci])
        repeat = min(max(1, int(round(w * (len(y) - n_pos) / max(n_pos, 1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(
            hidden_layer_sizes=CFG['mlp_hidden'], activation='relu',
            max_iter=200, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=42, learning_rate_init=5e-4, alpha=0.005,
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf
    print(f'  Trained {len(probe_models)} probes')
    return probe_models, scaler, pca, alpha_blend

def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.25):
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_act, above_med, score_rng = build_seq_features(scores_test[:, ci])
        X_test = np.hstack([Z_test, scores_test[:, ci:ci+1],
                            prev[:, None], next_[:, None],
                            mean[:, None], max_[:, None], std[:, None],
                            min_[:, None], rank[:, None],
                            p25[:, None], p75[:, None], win_pos[:, None], diff[:, None],
                            count_act[:, None], above_med[:, None], score_rng[:, None]])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print('MLP probes ready')

In [ ]:
# LightProtoSSM (identical to exp16)

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A)); self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x); x_ssm, _ = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2)
        x_c = F.silu(x_c)
        dt  = F.softplus(self.dt_proj(x_c))
        A   = -torch.exp(self.A_log)
        B_  = self.B_proj(x_c); C = self.C_proj(x_c)
        h   = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys  = []
        for t in range(T):
            h = h * torch.exp(A[None] * dt[:,t,:,None]) + x[:,t,:,None] * (dt[:,t,:,None] * B_[:,t,None,:])
            ys.append((h * C[:,t,None,:]).sum(-1))
        return torch.stack(ys, dim=1) + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16, n_ssm_layers=2,
                 n_classes=234, n_windows=N_WINDOWS, dropout=0.15,
                 n_sites=20, meta_dim=16, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.drop      = nn.Dropout(dropout)
        self.cross_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads=cross_attn_heads, dropout=dropout, batch_first=True)
            for _ in range(n_ssm_layers)])
        self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.prototypes  = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp  = nn.Parameter(torch.tensor(5.0))
        self.class_bias  = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta[:, None, :]
        for fwd, bwd, merge, norm, ca, cn in zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm,
                self.cross_attn, self.cross_norm):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            attn_out, _ = ca(h, h, h); h = cn(h + attn_out)
        h_n = F.normalize(h, dim=-1); p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1.0 - alpha) * perch_logits
        return sim


def train_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=30, patience=8, lr=5e-4, n_sites=20):
    n_files = len(emb_full) // N_WINDOWS
    emb_f = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    fnames  = meta_full.groupby('filename', sort=False).size().index.tolist()
    sites_u = sorted(meta_full['site'].astype(str).unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}
    site_ids = np.array([min(site2i.get(str(meta_full.loc[meta_full['filename']==fn,'site'].iloc[0]), 0), n_sites-1) for fn in fnames], dtype=np.int64)
    hour_ids = np.array([int(meta_full.loc[meta_full['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in fnames], dtype=np.int64)
    model = LightProtoSSM(d_model=CFG['d_model'], d_state=CFG['d_state'],
                          n_ssm_layers=CFG['n_ssm_layers'], n_classes=N_CLASSES,
                          n_windows=N_WINDOWS, dropout=CFG['dropout'],
                          n_sites=n_sites, meta_dim=CFG['meta_dim'])
    model.init_prototypes(torch.tensor(emb_full, dtype=torch.float32), torch.tensor(Y_full, dtype=torch.float32))
    emb_t  = torch.tensor(emb_f, dtype=torch.float32)
    log_t  = torch.tensor(log_f, dtype=torch.float32)
    lab_t  = torch.tensor(lab_f, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)
    pos_cnt    = lab_t.sum(dim=(0,1))
    pos_weight = ((lab_t.shape[0]*lab_t.shape[1] - pos_cnt) / (pos_cnt+1)).clamp(max=CFG['pos_weight_cap'])
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.3, anneal_strategy='cos')
    best_loss = float('inf'); best_state = None; wait = 0
    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (focal_bce(out, lab_t, gamma=CFG['focal_gamma'], pos_weight=pos_weight[None,None,:])
                + CFG['distill_weight'] * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
        if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval(); return model, site2i


def proto_predict(model, emb, scores, meta_df, site2i):
    n_files = len(emb) // N_WINDOWS
    fnames = meta_df.groupby('filename', sort=False).size().index.tolist()
    site_ids = np.array([
        min(site2i.get(str(meta_df.loc[meta_df['filename'] == fn, 'site'].iloc[0]), 0), CFG['n_sites'] - 1)
        for fn in fnames
    ], dtype=np.int64)
    hour_ids = np.array([
        int(meta_df.loc[meta_df['filename'] == fn, 'hour_utc'].iloc[0]) % 24
        for fn in fnames
    ], dtype=np.int64)
    with torch.no_grad():
        out = model(
            torch.tensor(emb.reshape(n_files, N_WINDOWS, -1), dtype=torch.float32),
            torch.tensor(scores.reshape(n_files, N_WINDOWS, -1), dtype=torch.float32),
            site_ids=torch.tensor(site_ids, dtype=torch.long),
            hours=torch.tensor(hour_ids, dtype=torch.long),
        ).numpy().reshape(-1, N_CLASSES)
    return out.astype(np.float32)


print('LightProtoSSM ready')

In [ ]:
# Final full-data training with Exp60 best configuration
LAM_EVENT = BEST_CFG['lambda_event']
LAM_TEXTURE = BEST_CFG['lambda_texture']
POWER_EVENT = BEST_CFG['power_event']
POWER_TEXTURE = BEST_CFG['power_texture']
ALPHA_EVENT = BEST_CFG['alpha_event']
ALPHA_TEXTURE = BEST_CFG['alpha_texture']
W_PROTO = BEST_CFG['wp']
W_MLP = BEST_CFG['wm']
W_PRIOR = BEST_CFG['wpr']

REQUIRED_HELPERS = ['train_proto_ssm', 'proto_predict', 'train_mlp_probes', 'apply_mlp_probes']
missing_helpers = [name for name in REQUIRED_HELPERS if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        'Notebook state is missing required helper definitions: ' + ', '.join(missing_helpers)
        + '. In Kaggle, run the notebook with Run All / Save Version so cells 1-8 execute before final training.'
    )

print('Training final ProtoSSM on all full-label standard windows')
proto_model, site2i = train_proto_ssm(
    emb_0, sc_0, Y_FULL_aligned, meta_tr.reset_index(drop=True),
    n_epochs=CFG['n_epochs'], patience=CFG['patience'], lr=CFG['lr'], n_sites=CFG['n_sites'],
)
print('Training final MLP probes on all full-label standard windows')
probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb_0, sc_0, Y_FULL_aligned,
    pca_dim=CFG['pca_dim'], alpha_blend=CFG['alpha_blend'], min_pos=CFG['min_pos'],
)

print('Final Exp60 config:', BEST_CFG)
print('Final training complete')


In [ ]:
# Test inference and submission writing
LAM_EVENT = BEST_CFG['lambda_event']
LAM_TEXTURE = BEST_CFG['lambda_texture']
POWER_EVENT = BEST_CFG['power_event']
POWER_TEXTURE = BEST_CFG['power_texture']
ALPHA_EVENT = BEST_CFG['alpha_event']
ALPHA_TEXTURE = BEST_CFG['alpha_texture']
W_PROTO = BEST_CFG['wp']
W_MLP = BEST_CFG['wm']
W_PRIOR = BEST_CFG['wpr']
TTA_W0 = BEST_CFG['tta_w0']


def infer_test_features(paths, offset_sec=0.0):
    return run_perch_windows(paths, offset_sec=offset_sec, batch_files=BATCH_FILES, verbose=True, row_id_suffix='')


def blend_logits(meta_df, emb, scores):
    prior_logits = apply_prior_taxon(
        scores,
        meta_df['site'].to_numpy(),
        meta_df['hour_utc'].to_numpy(),
        prior_tables,
        lambda_event=LAM_EVENT,
        lambda_texture=LAM_TEXTURE,
    )
    proto_logits = proto_predict(proto_model, emb, scores, meta_df, site2i)
    mlp_logits = apply_mlp_probes(emb, scores, probe_models, emb_scaler, emb_pca, alpha_blend)
    return (W_PROTO * proto_logits + W_MLP * mlp_logits + W_PRIOR * prior_logits).astype(np.float32)


def prior_only_submission():
    dry_meta = pd.DataFrame({'row_id': sample_sub['row_id']})
    stems = dry_meta['row_id'].str.rsplit('_', n=1).str[0] + '.ogg'
    parsed = stems.apply(parse_fname).apply(pd.Series)
    dry_meta = pd.concat([dry_meta, parsed], axis=1)
    dry_scores = np.zeros((len(dry_meta), N_CLASSES), dtype=np.float32)
    prior_logits = apply_prior_taxon(
        dry_scores,
        dry_meta['site'].to_numpy(),
        dry_meta['hour_utc'].to_numpy(),
        prior_tables,
        lambda_event=LAM_EVENT,
        lambda_texture=LAM_TEXTURE,
    )
    if len(dry_meta) % N_WINDOWS == 0:
        preds = postprocess_taxon_dual_power(prior_logits, power_event=POWER_EVENT, power_texture=POWER_TEXTURE, alpha_event=ALPHA_EVENT, alpha_texture=ALPHA_TEXTURE)
    else:
        # Kaggle's visible sample may contain only a few rows, so skip file-level
        # post-processing that requires complete 12-window soundscapes.
        preds = np.clip(sigmoid(prior_logits / temperatures[None, :]), 0.0, 1.0)
    out = pd.DataFrame(preds, columns=PRIMARY_LABELS)
    out.insert(0, 'row_id', sample_sub['row_id'].values)
    return out


test_paths = sorted(TEST_SOUNDSCAPE_DIR.glob('*.ogg')) if TEST_SOUNDSCAPE_DIR.exists() else []
print('Test audio files:', len(test_paths))

if test_paths:
    test_meta_0, test_scores_0, test_emb_0 = infer_test_features(test_paths, offset_sec=0.0)
    test_meta_250, test_scores_250, test_emb_250 = infer_test_features(test_paths, offset_sec=TTA_OFFSET_SEC)
    assert test_meta_0['row_id'].tolist() == test_meta_250['row_id'].tolist(), 'Standard and TTA row_id order differs'

    logits_0 = blend_logits(test_meta_0, test_emb_0, test_scores_0)
    logits_250 = blend_logits(test_meta_250, test_emb_250, test_scores_250)
    logits = TTA_W0 * logits_0 + (1.0 - TTA_W0) * logits_250
    preds = postprocess_taxon_dual_power(logits, power_event=POWER_EVENT, power_texture=POWER_TEXTURE, alpha_event=ALPHA_EVENT, alpha_texture=ALPHA_TEXTURE)

    pred_df = pd.DataFrame(preds, columns=PRIMARY_LABELS)
    pred_df.insert(0, 'row_id', test_meta_0['row_id'].values)
    submission = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
    missing = int(submission[PRIMARY_LABELS].isna().any(axis=1).sum())
    if missing:
        print(f'Warning: {missing} rows missing from audio inference; filling with global priors')
        for col in PRIMARY_LABELS:
            submission[col] = submission[col].fillna(prior_tables['global_p'][label_to_idx[col]])
else:
    print('No test audio found. Writing prior-only dry-run submission with sample_submission shape.')
    submission = prior_only_submission()

assert submission.shape == sample_sub.shape, (submission.shape, sample_sub.shape)
assert submission.columns.tolist() == sample_sub.columns.tolist()
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].clip(0.0, 1.0).astype(np.float32)
submission.to_csv(SUBMISSION_PATH, index=False)

summary = {
    'mode': 'test_audio' if test_paths else 'dry_run_no_test_audio',
    'submission_path': str(SUBMISSION_PATH),
    'shape': list(submission.shape),
    'test_files': len(test_paths),
    'mapped_perch_classes': int(MAPPED_MASK.sum()),
    'proxy_targets': len(proxy_map),
    'fitted_mlp_probes': len(probe_models),
    'best_cfg': BEST_CFG,
    'total_wall_minutes': round((time.time() - _WALL_START) / 60.0, 2),
}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
submission.head()


In [ ]:
# Exp60 validation grid search is intentionally omitted in this Kaggle submission notebook.
# The fixed config comes from exp60_taxon_dual_power and the models above are trained once on all available full-label training windows.
print('=' * 60)
print('Exp60 submit config:')
print('  MLP: PCA-128, (256,128), min_pos=3, alpha_blend=0.45')
print(f"  Taxon prior: lambda_event={BEST_CFG['lambda_event']}, lambda_texture={BEST_CFG['lambda_texture']}")
print(f"  Post-proc: power_event={BEST_CFG['power_event']}, power_texture={BEST_CFG['power_texture']}, alpha_event={BEST_CFG['alpha_event']}, alpha_texture={BEST_CFG['alpha_texture']}")
print(f"  Ensemble: wp={BEST_CFG['wp']}, wm={BEST_CFG['wm']}, wpr={BEST_CFG['wpr']}, tta_w0={BEST_CFG['tta_w0']}")
print('  Validation: AUC=0.96644  cmAP=0.95742')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
